In [3]:
import openai, json

client = openai.OpenAI()
messages = []

In [4]:
def get_weather(city):
    return f"{city}의 온도는 33도입니다"

FUNCTION_MAP = {
    'get_weather': get_weather
}

In [5]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description":"A function to get the weather of a city.",
            "parameters":{
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city to get the weather of."
                    }
                },
                "required": ["city"]
            }

        }
    }
]

In [10]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments
                        }
                    } for tool_call in message.tool_calls
                ]
        })

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            print(f"Ran {function_name} with args {arguments} for a result of {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-5-nano", messages=messages, tools=TOOLS
    )
    process_ai_response(response.choices[0].message)

In [11]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({
            "role": "user",
            "content": message
        })
        print(f"User: {message}")
        call_ai()


User: 마드리드의 날씨는 어때?
AI: 수원이님, 마드리드의 현재 날씨는 33도예요. 매우 더운 편이니 물 많이 마시고 자외선 차단제나 모자 등으로 대비하시길 권해요. 더 자세한 정보가 필요하시면 습도나 바람, 비 소식 같은 것도 확인해 드릴게요. 어떤 항목을 알려드릴까요?
User: 수원의 날씨는 어때?
Calling function: get_weather with {"city":"Suwon"}
Ran get_weather with args {'city': 'Suwon'} for a result of Suwon의 온도는 33도입니다
AI: 수원이님, 수원 현재 날씨는 33도예요. 매우 더운 편이니 물 많이 마시고 자외선 차단제와 모자 등으로 대비하세요. 필요하시면 습도, 바람, 비 소식 등 자세한 정보도 확인해 드릴게요. 어떤 정보를 원하시나요?
